# Fase 4: tiempos de memoria constante frente a global

Mide `build/hough` (global) y `build/hough_const` (global + constante) en la **misma sesión y GPU**, con CUDA Events alrededor de cada lanzamiento y validación contra la CPU en cada ejecución.

Dos pruebas previas mostraron que la frecuencia de la GPU cambia durante la sesión y que ese efecto supera la diferencia entre versiones: primero subía al salir de reposo (`results/fase4-constante/exploratorio/`) y después bajaba porque la GPU pasa casi todo el tiempo sin trabajo entre lanzamientos cortos (`results/fase4-constante/frecuencia-variable/`). Por eso aquí:

1. se estabiliza la GPU con 2000 ejecuciones descartadas;
2. cada ejecución hace **100 lanzamientos medidos**; los primeros 90 mantienen ocupada la GPU y se reportan los **últimos 10**;
3. se alternan cuatro rondas en orden ABBA (global-constante, constante-global, ...) y se registra el estado de la GPU antes y después de cada ejecución.

Usa **Entorno de ejecución → Cambiar tipo de entorno → GPU T4**.

In [ ]:
!nvidia-smi
!nvcc --version

## Proyecto

In [ ]:
from pathlib import Path
import subprocess, os
RAMA = 'main'
root = Path('/content/P3-CUDA-Paralela')
if not root.exists():
    subprocess.run(['git', 'clone', '--branch', RAMA,
                    'https://github.com/Jorge162017/P3-CUDA-Paralela.git', str(root)], check=True)
os.chdir(root)
subprocess.run(['git', 'fetch', 'origin', RAMA], check=True)
subprocess.run(['git', 'checkout', RAMA], check=True)
subprocess.run(['git', 'pull', '--ff-only', 'origin', RAMA], check=True)
subprocess.run(['make', 'build/hough', 'build/hough_const'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Rama:', RAMA, '| Commit:', commit)

## Cien lanzamientos por ejecución; se reportan los últimos diez

Cada ejecución escribe su CSV completo (100 filas) en `build/`. A `results/fase4-constante/final/` se copian solo las 10 mediciones reportadas (91 a 100), con las mismas columnas que la bitácora global. El log resume cada ejecución con el promedio de los lanzamientos 1–10, 11–90 y 91–100, para ver si la GPU se estabilizó, y se guarda aunque falle una ejecución.

In [ ]:
import shutil
import pandas as pd
out = Path('results/fase4-constante/final')
LANZAMIENTOS, REPORTADOS = 100, 10
shutil.rmtree(out, ignore_errors=True)
out.mkdir(parents=True)
ORDEN = {'ronda1': ['global', 'constante'], 'ronda2': ['constante', 'global'],
         'ronda3': ['global', 'constante'], 'ronda4': ['constante', 'global']}
PROGRAMA = {'global': './build/hough', 'constante': './build/hough_const'}
CONSULTA = ['nvidia-smi', '--query-gpu=pstate,clocks.sm,clocks.mem,temperature.gpu,power.draw',
            '--format=csv,noheader,nounits']
log = subprocess.check_output(['nvidia-smi'], text=True)
log += subprocess.check_output(['nvcc', '--version'], text=True)
log += f'Rama: {RAMA}\nCommit: {commit}\n'
log += 'Compilacion: make build/hough build/hough_const; -O2 -std=c++17 --fmad=false\n'
log += 'Protocolo: 2000 ejecuciones descartadas para estabilizar la GPU; luego 4 rondas ABBA, '
log += f'cada ejecucion con 1 calentamiento + {LANZAMIENTOS} mediciones; se reportan las ultimas {REPORTADOS}\n'
estabilizar = subprocess.run([PROGRAMA['global'], 'data/runway.pgm', '--runs', '2000'],
                             text=True, capture_output=True)
log += f'== estabilizacion (descartada): codigo {estabilizar.returncode}; '
log += estabilizar.stdout.splitlines()[-2] if estabilizar.returncode == 0 else estabilizar.stderr
log += '\n'
assert estabilizar.returncode == 0, estabilizar.stderr
tablas, estados = [], []
try:
    for ronda, variantes in ORDEN.items():
        for variante in variantes:
            antes = subprocess.check_output(CONSULTA, text=True).strip()
            temporal = Path('build/fase4') / ronda / variante
            shutil.rmtree(temporal, ignore_errors=True)
            run = subprocess.run([PROGRAMA[variante], 'data/runway.pgm', '--runs', str(LANZAMIENTOS),
                                  '--output', str(temporal)], text=True, capture_output=True)
            despues = subprocess.check_output(CONSULTA, text=True).strip()
            estados += [f'{ronda},{variante},antes,{antes}', f'{ronda},{variante},despues,{despues}']
            resumen_programa = '\n'.join(l for l in run.stdout.splitlines() if not l.startswith('Ejecucion'))
            salida = f'== {ronda} {variante} | GPU antes: {antes} | despues: {despues}\n{resumen_programa}\n{run.stderr}'
            print(salida)
            log += salida
            assert run.returncode == 0, f'Fallo {variante} en {ronda}'
            completa = pd.read_csv(temporal / f'{variante}-times.csv')
            assert len(completa) == LANZAMIENTOS and completa['validated'].all()
            assert completa['run'].tolist() == list(range(1, LANZAMIENTOS + 1))
            tramos = {'1-10': completa['run'] <= 10, '11-90': completa['run'].between(11, 90),
                      '91-100': completa['run'] > LANZAMIENTOS - REPORTADOS}
            log += 'Promedios por tramo (ms): ' + '; '.join(
                f"{nombre}: {completa.loc[filtro, 'kernel_ms'].mean():.6f}" for nombre, filtro in tramos.items()) + '\n'
            # Copia literal de la cabecera y las ultimas filas, con el formato del programa.
            lineas = (temporal / f'{variante}-times.csv').read_text().splitlines()
            destino = out / ronda / f'{variante}-times.csv'
            destino.parent.mkdir(exist_ok=True)
            destino.write_text('\n'.join([lineas[0], *lineas[-REPORTADOS:]]) + '\n')
            reportada = pd.read_csv(destino)
            assert reportada['run'].tolist() == list(range(LANZAMIENTOS - REPORTADOS + 1, LANZAMIENTOS + 1))
            tablas.append(reportada.assign(ronda=ronda))
finally:
    (out / 'fase4-validacion.txt').write_text(log)
    (out / 'estado-gpu.csv').write_text('ronda,variant,momento,pstate,sm_mhz,mem_mhz,temp_c,power_w\n'
                                       + '\n'.join(estados) + '\n')
tiempos = pd.concat(tablas, ignore_index=True)
display(tiempos.pivot_table(index='run', columns=['ronda', 'variant'], values='kernel_ms'))

## Comparación

Desviación estándar muestral, igual que en la fase 2. `diferencia_pct` es negativa si la versión constante es más rápida; `speedup` es tiempo global / tiempo constante. `p_welch` prueba si la diferencia de promedios es estadísticamente significativa. Solo se usan las ejecuciones 91 a 100 de cada CSV; la fila `total` combina esas 40 mediciones por versión.

In [ ]:
resumen = tiempos.groupby(['ronda', 'variant'])['kernel_ms'].agg(['count', 'mean', 'std', 'min', 'max'])
total = tiempos.groupby('variant')['kernel_ms'].agg(['count', 'mean', 'std', 'min', 'max'])
resumen = pd.concat([resumen, pd.concat({'total': total})])
resumen.to_csv(out / 'resumen.csv')
display(resumen)
filas = []
for ronda in [*ORDEN, 'total']:
    datos = tiempos if ronda == 'total' else tiempos.query('ronda == @ronda')
    g = datos.query('variant == "global"')['kernel_ms']
    c = datos.query('variant == "constante"')['kernel_ms']
    fila = dict(ronda=ronda, global_ms=g.mean(), constante_ms=c.mean(),
                diferencia_pct=100 * (c.mean() - g.mean()) / g.mean(), speedup=g.mean() / c.mean())
    try:
        from scipy.stats import ttest_ind
        fila['p_welch'] = ttest_ind(g, c, equal_var=False).pvalue
    except ImportError:
        pass
    filas.append(fila)
comparacion = pd.DataFrame(filas)
comparacion.to_csv(out / 'comparacion.csv', index=False)
display(comparacion)
display(pd.read_csv(out / 'estado-gpu.csv'))

## Resultados

In [ ]:
from google.colab import files
import zipfile
bundle = Path('build/fase4-constante.zip')
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(out.rglob('*')):
        if path.is_file():
            archive.write(path, path.as_posix())
files.download(str(bundle))